Day 1: Data Ingestion

In [1]:
import os
import requests
import pandas as pd

# The 6 mutual fund schemes we need to capture live data for
schemes = {
    125497: "HDFC_Top_100_Direct",
    119551: "SBI_Bluechip",
    120503: "ICICI_Bluechip",
    118632: "Nippon_Large_Cap",
    119092: "Axis_Bluechip",
    120841: "Kotak_Bluechip"
}

# Loop over each scheme, pull data from the web, and save it
for code, name in schemes.items():
    url = f"https://api.mfapi.in/mf/{code}"
    print(f"Fetching data for {name} ({code})...")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        raw_json = response.json()
        
        # Turn the historical price records into a neat table
        df = pd.DataFrame(raw_json['data'])
        
        # Track metadata parameters for clarity
        df['scheme_code'] = code
        df['scheme_name'] = raw_json['meta']['scheme_name']
        
        # Save structured CSV file back to your raw data folder
        csv_path = f"data/raw/{name}_nav.csv"
        df.to_csv(csv_path, index=False)
        print(f"--> Successfully saved to {csv_path}\n")
    else:
        print(f"[ERROR] Failed to reach API for {name}. Status code: {response.status_code}\n")

Fetching data for HDFC_Top_100_Direct (125497)...
--> Successfully saved to data/raw/HDFC_Top_100_Direct_nav.csv

Fetching data for SBI_Bluechip (119551)...
--> Successfully saved to data/raw/SBI_Bluechip_nav.csv

Fetching data for ICICI_Bluechip (120503)...
--> Successfully saved to data/raw/ICICI_Bluechip_nav.csv

Fetching data for Nippon_Large_Cap (118632)...
--> Successfully saved to data/raw/Nippon_Large_Cap_nav.csv

Fetching data for Axis_Bluechip (119092)...
--> Successfully saved to data/raw/Axis_Bluechip_nav.csv

Fetching data for Kotak_Bluechip (120841)...
--> Successfully saved to data/raw/Kotak_Bluechip_nav.csv



In [2]:
import pandas as pd
import glob
import os

print("=========================================")
print("  LOADING AND PROFILING ALL RAW DATA  ")
print("=========================================\n")

# Look inside data/raw and find all CSV files automatically
csv_files = glob.glob("data/raw/*.csv")

if not csv_files:
    print("[ALERT] No CSV files found in data/raw/. Please check your folders.")
else:
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        df = pd.read_csv(file_path)
        
        print(f"📁 Dataset Name: {filename}")
        print(f"   • Data Dimensions (Rows, Columns): {df.shape}")
        print(f"   • Missing Values Detected: {df.isnull().sum().sum()}")
        print(f"   • First 2 Rows:")
        print(df.head(2))
        print("-" * 50)

print("\n=========================================")
print("       MUTUAL FUND QUALITY CHECK         ")
print("=========================================\n")

# Cross-checking the structural master file if it exists
master_path = "data/raw/fund_master.csv"
history_path = "data/raw/nav_history.csv"

if os.path.exists(master_path) and os.path.exists(history_path):
    master_df = pd.read_csv(master_path)
    history_df = pd.read_csv(history_path)
    
    # 1. Explore Master Categorization Pools
    print("📋 Fund Master Properties:")
    if 'fund_house' in master_df.columns:
        print(f"   • Total Unique Fund Houses: {master_df['fund_house'].nunique()}")
    if 'category' in master_df.columns:
        print(f"   • Total Unique Categories: {master_df['category'].nunique()}")
        
    # 2. Validate Code Integrities
    if 'scheme_code' in master_df.columns and 'scheme_code' in history_df.columns:
        master_codes = set(master_df['scheme_code'].unique())
        history_codes = set(history_df['scheme_code'].unique())
        
        missing = master_codes - history_codes
        
        print("\n🔍 Referential Integrity Summary:")
        if len(missing) == 0:
            print("   • Success: Every scheme code in fund_master matches historical pricing data.")
        else:
            print(f"   • Warning: {len(missing)} codes in fund_master do not exist in your history files.")
else:
    print("ℹ️ Note: Place 'fund_master.csv' and 'nav_history.csv' in data/raw/ to run the structural integrity cross-checks.")

  LOADING AND PROFILING ALL RAW DATA  

📁 Dataset Name: Axis_Bluechip_nav.csv
   • Data Dimensions (Rows, Columns): (3579, 4)
   • Missing Values Detected: 0
   • First 2 Rows:
         date        nav  scheme_code  \
0  19-06-2026  6195.7815       119092   
1  18-06-2026  6194.6520       119092   

                                         scheme_name  
0  HDFC Money Market Fund - Growth Option - Direc...  
1  HDFC Money Market Fund - Growth Option - Direc...  
--------------------------------------------------
📁 Dataset Name: HDFC_Top_100_Direct_nav.csv
   • Data Dimensions (Rows, Columns): (3105, 4)
   • Missing Values Detected: 0
   • First 2 Rows:
         date       nav  scheme_code  \
0  19-06-2026  202.0761       125497   
1  18-06-2026  200.9565       125497   

                                 scheme_name  
0  SBI Small Cap Fund - Direct Plan - Growth  
1  SBI Small Cap Fund - Direct Plan - Growth  
--------------------------------------------------
📁 Dataset Name: ICICI_Bluec

In [5]:
import os
import pandas as pd

print("==================================================")
print("     STAGE 1: PROFILING ALL 10 BASE DATASETS      ")
print("==================================================\n")

base_files = {
    "01_fund_master.csv": "Fund Master",
    "02_nav_history.csv": "NAV History",
    "03_aum_by_fund_house.csv": "AUM by Fund House",
    "04_monthly_sip_inflows.csv": "Monthly SIP Inflows",
    "05_category_inflows.csv": "Category Inflows",
    "06_industry_folio_count.csv": "Industry Folio Count",
    "07_scheme_performance.csv": "Scheme Performance",
    "08_investor_transactions.csv": "Investor Transactions",
    "09_portfolio_holdings.csv": "Portfolio Holdings",
    "10_benchmark_indices.csv": "Benchmark Indices"
}

# Remember your notebook is inside notebooks/, so we go up one level to find data/
for filename, logical_name in base_files.items():
    path = os.path.join("..", "data", "raw", filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"📋 Dataset: {logical_name} ({filename})")
        print(f"   • Dimensions (Rows, Columns): {df.shape}")
        print(f"   • Missing Values: {df.isnull().sum().sum()}")
        print(f"   • Columns & Dtypes:")
        for col, dtype in df.dtypes.items():
            print(f"     - {col}: {dtype}")
        print("-" * 50)
    else:
        print(f"⚠️ Cannot find {path}. Check file names.")

print("\n==================================================")
print("     STAGE 2: INTEGRITY & CATEGORY EXPLORATION    ")
print("==================================================\n")

master_path = os.path.join("..", "data", "raw", "01_fund_master.csv")
history_path = os.path.join("..", "data", "raw", "02_nav_history.csv")

if os.path.exists(master_path) and os.path.exists(history_path):
    m_df = pd.read_csv(master_path)
    h_df = pd.read_csv(history_path)
    
    cat_cols = [c for c in m_df.columns if 'category' in c.lower()]
    house_cols = [c for c in m_df.columns if 'house' in c.lower() or 'amc' in c.lower()]
    
    if house_cols:
        print(f"   • Unique Fund Houses: {m_df[house_cols[0]].nunique()}")
    if cat_cols:
        print(f"   • Unique Fund Categories: {m_df[cat_cols[0]].nunique()}")
        
    m_code = [c for c in m_df.columns if 'code' in c.lower()]
    h_code = [c for c in h_df.columns if 'code' in c.lower()]
    
    if m_code and h_code:
        unmatched = set(m_df[m_code[0]].unique()) - set(h_df[h_code[0]].unique())
        print("\n🔍 Referential Code Audit:")
        if not unmatched:
            print("   • SUCCESS: Every scheme code in Fund Master matches historical files.")
        else:
            print(f"   • AMFI ALERT: {len(unmatched)} master codes don't have historical entries.")

     STAGE 1: PROFILING ALL 10 BASE DATASETS      

📋 Dataset: Fund Master (01_fund_master.csv)
   • Dimensions (Rows, Columns): (40, 15)
   • Missing Values: 0
   • Columns & Dtypes:
     - amfi_code: int64
     - fund_house: object
     - scheme_name: object
     - category: object
     - sub_category: object
     - plan: object
     - launch_date: object
     - benchmark: object
     - expense_ratio_pct: float64
     - exit_load_pct: float64
     - min_sip_amount: int64
     - min_lumpsum_amount: int64
     - fund_manager: object
     - risk_category: object
     - sebi_category_code: object
--------------------------------------------------
📋 Dataset: NAV History (02_nav_history.csv)
   • Dimensions (Rows, Columns): (46000, 3)
   • Missing Values: 0
   • Columns & Dtypes:
     - amfi_code: int64
     - date: object
     - nav: float64
--------------------------------------------------
📋 Dataset: AUM by Fund House (03_aum_by_fund_house.csv)
   • Dimensions (Rows, Columns): (90, 5)
 

In [6]:
import os
import requests
import pandas as pd

# Target scheme configurations
SCHEMES = {
    125497: "HDFC_Top_100_Direct",
    119551: "SBI_Bluechip",
    120503: "ICICI_Bluechip",
    118632: "Nippon_Large_Cap",
    119092: "Axis_Bluechip",
    120841: "Kotak_Bluechip"
}

def fetch_live_nav():
    print("==================================================")
    print("          STARTING LIVE NAV INGESTION             ")
    print("==================================================\n")
    
    # Target directory for raw files
    raw_dir = os.path.join("data", "raw")
    os.makedirs(raw_dir, exist_ok=True)
    
    for code, name in SCHEMES.items():
        url = f"https://api.mfapi.in/mf/{code}"
        print(f"📡 Fetching live data for {name} ({code})...")
        
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            raw_json = response.json()
            
            # Convert series to tabular DataFrame
            df = pd.DataFrame(raw_json['data'])
            
            # Enrich records with essential metadata
            df['scheme_code'] = code
            df['scheme_name'] = raw_json['meta'].get('scheme_name', name)
            df['isin_growth'] = raw_json['meta'].get('isin_div_payer_isin_growth', None)
            
            # Save flattened output
            csv_path = os.path.join(raw_dir, f"{name}_nav.csv")
            df.to_csv(csv_path, index=False)
            print(f"✔️ Successfully saved to {csv_path} ({len(df)} rows)\n")
            
        except Exception as e:
            print(f"❌ Error fetching {name}: {e}\n")

if __name__ == "__main__":
    fetch_live_nav()

          STARTING LIVE NAV INGESTION             

📡 Fetching live data for HDFC_Top_100_Direct (125497)...
✔️ Successfully saved to data\raw\HDFC_Top_100_Direct_nav.csv (3105 rows)

📡 Fetching live data for SBI_Bluechip (119551)...
✔️ Successfully saved to data\raw\SBI_Bluechip_nav.csv (3250 rows)

📡 Fetching live data for ICICI_Bluechip (120503)...
✔️ Successfully saved to data\raw\ICICI_Bluechip_nav.csv (3321 rows)

📡 Fetching live data for Nippon_Large_Cap (118632)...
✔️ Successfully saved to data\raw\Nippon_Large_Cap_nav.csv (3312 rows)

📡 Fetching live data for Axis_Bluechip (119092)...
✔️ Successfully saved to data\raw\Axis_Bluechip_nav.csv (3579 rows)

📡 Fetching live data for Kotak_Bluechip (120841)...
✔️ Successfully saved to data\raw\Kotak_Bluechip_nav.csv (3315 rows)



In [12]:
!git add .
!git commit -m "docs: save Day 1 technical accomplishments report"

[master 1322802] docs: save Day 1 technical accomplishments report
 1 file changed, 342 insertions(+), 1 deletion(-)


In [14]:
!git add ..
!git commit -m "Day 1: Data ingestion complete"

[master 4acb80c] Day 1: Data ingestion complete
 1 file changed, 1 insertion(+)
 create mode 100644 reports/Data Quality Summary Report.md


In [ ]:
!git add ..
!git commit -m "docs: update summary with scheme name anomaly findings"